In [2]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=3f1ab51063f19c0ba2bb136796494e63d986dbb8786b9b3f6993a75130884ad2
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [40]:
import wikipedia

def link_entity(name):
    try:
        page = wikipedia.page(name)
        return page.title, page.url
    except:
        return None

print(link_entity("Narendra modi"))

('Narendra Modi', 'https://en.wikipedia.org/wiki/Narendra_Modi')


In [20]:
!pip install spacy requests SPARQLWrapper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 28.4 MB/s eta 0:00:00


In [21]:
!python -m spacy download de_core_news_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.8/567.8 MB 1.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [26]:
"""

pip install spacy requests SPARQLWrapper

German model:
python -m spacy download de_core_news_lg

English model:
python -m spacy download en_core_web_lg

"""

import os
import json
import time
import atexit
import requests
import spacy

from SPARQLWrapper import SPARQLWrapper, JSON

# ============================================================
# CONFIG
# ============================================================

WIKIDATA_API = "https://www.wikidata.org/w/api.php"

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"

SPACY_MODEL = "de_core_news_lg"

LANGUAGE = "de"

REQUEST_DELAY = 0.15

HEADERS = {
    "User-Agent": "WikidataMasker/1.0 (research-project)"
}

ENTITY_CACHE_FILE = "entity_cache.json"

ONTOLOGY_CACHE_FILE = "ontology_cache.json"

VALID_ENTITY_LABELS = {
    "PER",
    "ORG",
    "LOC",
    "MISC"
}

# ============================================================
# LOAD SPACY
# ============================================================

print("\nLoading spaCy model...")

nlp = spacy.load(SPACY_MODEL)

# ============================================================
# SPARQL CLIENT
# ============================================================

sparql = SPARQLWrapper(SPARQL_ENDPOINT)

# ============================================================
# CACHE MANAGER
# ============================================================

class CacheManager:

    def __init__(self, path):

        self.path = path

        self.cache = self.load()

    def load(self):

        if os.path.exists(self.path):

            try:

                with open(
                    self.path,
                    "r",
                    encoding="utf-8"
                ) as f:

                    return json.load(f)

            except:

                return {}

        return {}

    def save(self):

        with open(
            self.path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                self.cache,
                f,
                ensure_ascii=False,
                indent=2
            )

    def get(self, key):

        return self.cache.get(key)

    def set(self, key, value):

        self.cache[key] = value

# ============================================================
# GLOBAL CACHES
# ============================================================

entity_cache = CacheManager(
    ENTITY_CACHE_FILE
)

ontology_cache = CacheManager(
    ONTOLOGY_CACHE_FILE
)

# ============================================================
# AUTO SAVE
# ============================================================

@atexit.register
def save_all_caches():

    entity_cache.save()

    ontology_cache.save()

    print("\nCaches saved.")

# ============================================================
# NER EXTRACTOR
# ============================================================

class NERExtractor:

    def __init__(self):

        self.nlp = nlp

    def extract(self, text):

        doc = self.nlp(text)

        entities = []

        for ent in doc.ents:

            # ====================================================
            # FILTER INVALID LABELS
            # ====================================================

            if ent.label_ not in VALID_ENTITY_LABELS:
                continue

            # ====================================================
            # FILTER LOWERCASE NOISE
            # ====================================================

            if ent.text.islower():
                continue

            # ====================================================
            # FILTER VERY SHORT TOKENS
            # ====================================================

            if len(ent.text.strip()) < 3:
                continue

            entities.append({

                "text": ent.text,

                "label": ent.label_,

                "start": ent.start_char,

                "end": ent.end_char
            })

        return entities

# ============================================================
# ENTITY LINKER
# ============================================================

class EntityLinker:

    def resolve(self, entity_text):

        # ========================================================
        # CACHE HIT
        # ========================================================

        cached = entity_cache.get(entity_text)

        if cached:

            return cached

        # ========================================================
        # THROTTLING
        # ========================================================

        time.sleep(REQUEST_DELAY)

        params = {

            "action": "wbsearchentities",

            "search": entity_text,

            "language": LANGUAGE,

            "format": "json",

            "limit": 1
        }

        try:

            response = requests.get(

                WIKIDATA_API,

                params=params,

                headers=HEADERS,

                timeout=10
            )

            # ====================================================
            # HTTP ERROR
            # ====================================================

            if response.status_code != 200:

                print(
                    f"[HTTP ERROR] "
                    f"{response.status_code} "
                    f"→ {entity_text}"
                )

                return None

            # ====================================================
            # SAFE JSON
            # ====================================================

            try:

                data = response.json()

            except:

                print(
                    f"[JSON ERROR] "
                    f"{entity_text}"
                )

                print(
                    response.text[:200]
                )

                return None

            # ====================================================
            # NO RESULT
            # ====================================================

            if not data.get("search"):

                print(
                    f"[NO MATCH] "
                    f"{entity_text}"
                )

                return None

            result = data["search"][0]

            entity_data = {

                "qid": result.get("id"),

                "label": result.get("label"),

                "description": result.get("description")
            }

            entity_cache.set(
                entity_text,
                entity_data
            )

            return entity_data

        except requests.exceptions.Timeout:

            print(
                f"[TIMEOUT] "
                f"{entity_text}"
            )

            return None

        except requests.exceptions.ConnectionError:

            print(
                f"[CONNECTION ERROR] "
                f"{entity_text}"
            )

            return None

        except Exception as e:

            print(
                f"[EntityLinker ERROR] "
                f"{entity_text} → {e}"
            )

            return None

# ============================================================
# WIKIDATA ONTOLOGY FETCHER
# ============================================================
class WikidataOntologyFetcher:

    ENTITY_URL = (
        "https://www.wikidata.org/wiki/Special:EntityData/"
    )

    PROPERTY_MAP = {

        "P106": "occupations",

        "P31": "instances"
    }

    def fetch(self, qid):

        # ====================================================
        # CACHE HIT
        # ====================================================

        cached = ontology_cache.get(qid)

        if cached:
            return cached

        time.sleep(REQUEST_DELAY)

        url = f"{self.ENTITY_URL}{qid}.json"

        try:

            response = requests.get(

                url,

                headers=HEADERS,

                timeout=10
            )

            if response.status_code != 200:

                print(
                    f"[HTTP ERROR] "
                    f"{response.status_code} "
                    f"→ {qid}"
                )

                return self.empty()

            data = response.json()

            entity = data["entities"][qid]

            claims = entity.get("claims", {})

            ontology = {

                "occupations": [],

                "instances": []
            }

            # =================================================
            # EXTRACT CLAIMS
            # =================================================

            for prop, target in self.PROPERTY_MAP.items():

                if prop not in claims:
                    continue

                values = []

                for item in claims[prop]:

                    try:

                        value_qid = item[
                            "mainsnak"
                        ][
                            "datavalue"
                        ][
                            "value"
                        ][
                            "id"
                        ]

                        label = self.fetch_label(
                            value_qid
                        )

                        if label:
                            values.append(
                                label.lower()
                            )

                    except:
                        pass

                ontology[target] = values

            ontology_cache.set(
                qid,
                ontology
            )

            return ontology

        except Exception as e:

            print(
                f"[Ontology ERROR] "
                f"{qid} → {e}"
            )

            return self.empty()

    def fetch_label(self, qid):

        cached = ontology_cache.get(
            f"label::{qid}"
        )

        if cached:
            return cached

        url = f"{self.ENTITY_URL}{qid}.json"

        try:

            response = requests.get(
                url,
                headers=HEADERS,
                timeout=10
            )

            data = response.json()

            entity = data["entities"][qid]

            labels = entity.get("labels", {})

            label = labels.get(
                LANGUAGE,
                {}
            ).get("value")

            if label:

                ontology_cache.set(
                    f"label::{qid}",
                    label
                )

            return label

        except:

            return None

    def empty(self):

        return {

            "occupations": [],

            "instances": []
        }

ROLE_RULES = {

    # ========================================================
    # PEOPLE
    # ========================================================

    "film_actor": {

        "actor",
        "film actor",
        "television actor",
        "actress"
    },

    "politician": {

        "politician",
        "president",
        "minister",
        "chancellor",
        "member of parliament"
    },

    "business_person": {

        "businessperson",
        "entrepreneur",
        "ceo",
        "founder"
    },

    "scientist": {

        "scientist",
        "researcher",
        "physicist",
        "mathematician"
    },

    "musician": {

        "musician",
        "singer",
        "composer"
    },

    "athlete": {

        "athlete",
        "football player",
        "tennis player",
        "basketball player"
    },

    "journalist": {

        "journalist",
        "reporter"
    },

    "religious_leader": {

        "pope",
        "imam",
        "priest",
        "rabbi"
    },

    # ========================================================
    # ORGANIZATIONS
    # ========================================================

    "company": {

        "company",
        "business",
        "corporation",
        "enterprise"
    },

    "educational_institution": {

        "university",
        "college",
        "school",
        "educational institution"
    },

    "government_agency": {

        "government agency",
        "ministry",
        "public authority"
    },

    "research_institute": {

        "research institute"
    },

    "sports_club": {

        "sports club",
        "football club"
    },

    # ========================================================
    # LOCATIONS
    # ========================================================

    "city": {

        "city",
        "capital city"
    },

    "country": {

        "country",
        "sovereign state"
    },

    "administrative_region": {

        "state",
        "province",
        "region"
    },

    "continent": {

        "continent"
    },

    "building": {

        "building",
        "skyscraper"
    },

    "venue": {

        "stadium",
        "arena"
    },

    "transport_hub": {

        "airport",
        "train station"
    },

    "water_body": {

        "river",
        "lake",
        "sea",
        "ocean"
    }
}
class OntologyNormalizer:

    def normalize(self, ontology):

        concepts = set()

        concepts.update(
            ontology.get("occupations", [])
        )

        concepts.update(
            ontology.get("instances", [])
        )

        concepts.update(
            ontology.get("subclasses", [])
        )

        roles = set()

        for role, vocab in ROLE_RULES.items():

            if concepts & vocab:

                roles.add(role)

        if not roles:

            return ["entity"]

        return list(roles)

# ============================================================
# SEMANTIC MAPPER
# ============================================================

class SemanticMapper:

    def __init__(self):

        self.fetcher = WikidataOntologyFetcher()

        self.normalizer = OntologyNormalizer()

    def resolve_roles(self, qid):

        ontology = self.fetcher.fetch(qid)

        roles = self.normalizer.normalize(
            ontology
        )

        return {

            "ontology": ontology,

            "roles": roles
        }

# ============================================================
# ENTITY MASKER
# ============================================================

class EntityMasker:

    def __init__(self):

        self.ner = NERExtractor()

        self.linker = EntityLinker()

        self.mapper = SemanticMapper()

    def build_replacement(

        self,

        entity,

        mode="hybrid"
    ):

        entity_text = entity["text"]

        print(
            f"\nResolving: "
            f"{entity_text}"
        )

        linked = self.linker.resolve(
            entity_text
        )

        # ========================================================
        # LINK FAILURE
        # ========================================================

        if not linked:

            return "[ENTITY]"

        qid = linked["qid"]

        semantic = self.mapper.resolve_roles(
            qid
        )

        roles = semantic["roles"]

        role = roles[0]

        print(f"QID: {qid}")

        print(f"Roles: {roles}")

        # ========================================================
        # MODE: ROLE
        # ========================================================

        if mode == "role":

            return role

        # ========================================================
        # MODE: NER
        # ========================================================

        elif mode == "ner":

            return f"[{entity['label']}]"

        # ========================================================
        # MODE: HYBRID
        # ========================================================

        else:

            if role == "entity":

                return f"[{entity['label']}]"

            return role

    def mask(

        self,

        text,

        mode="hybrid"
    ):

        entities = self.ner.extract(text)

        if not entities:

            return text

        # ========================================================
        # SORT ENTITIES BY START POSITION
        # ========================================================

        entities = sorted(
            entities,
            key=lambda x: x["start"]
        )

        pieces = []

        last_idx = 0

        for ent in entities:

            start = ent["start"]

            end = ent["end"]

            # ====================================================
            # ORIGINAL TEXT BEFORE ENTITY
            # ====================================================

            pieces.append(
                text[last_idx:start]
            )

            # ====================================================
            # ENTITY REPLACEMENT
            # ====================================================

            replacement = self.build_replacement(
                ent,
                mode
            )

            pieces.append(
                replacement
            )

            last_idx = end

        # ========================================================
        # REMAINING TEXT
        # ========================================================

        pieces.append(
            text[last_idx:]
        )

        output = "".join(pieces)

        return output

# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    masker = EntityMasker()

    texts = [

        "Brad Pitt spielte in vielen Hollywood Filmen.",

        "Angela Merkel traf Elon Musk in Berlin.",

        "BMW ist ein deutsches Unternehmen.",

        "Harvard University ist weltbekannt.",

        "Lionel Messi gewann viele Fußballtitel.",

        "Papst Franziskus besuchte Deutschland.",

        "Amazon investiert in Berlin.",

        "Barack Obama traf Angela Merkel."
    ]

    print("\n================================================")
    print("WIKIDATA KB-GROUNDED ENTITY MASKER")
    print("================================================")

    for text in texts:

        print("\n------------------------------------------------")
        print("ORIGINAL:")
        print(text)

        masked = masker.mask(
            text,
            mode="hybrid"
        )

        print("\nMASKED:")
        print(masked)


Loading spaCy model...

WIKIDATA KB-GROUNDED ENTITY MASKER

------------------------------------------------
ORIGINAL:
Brad Pitt spielte in vielen Hollywood Filmen.

Resolving: Brad Pitt
QID: Q35332
Roles: ['film_actor']

Resolving: Hollywood
QID: Q34006
Roles: ['entity']

MASKED:
film_actor spielte in vielen [LOC] Filmen.

------------------------------------------------
ORIGINAL:
Angela Merkel traf Elon Musk in Berlin.

Resolving: Angela Merkel
QID: Q567
Roles: ['scientist', 'politician']

Resolving: Elon Musk
QID: Q317521
Roles: ['business_person', 'politician']

Resolving: Berlin
QID: Q64
Roles: ['entity']

MASKED:
scientist traf business_person in [LOC].

------------------------------------------------
ORIGINAL:
BMW ist ein deutsches Unternehmen.

Resolving: BMW
QID: Q26678
Roles: ['company']

MASKED:
company ist ein deutsches Unternehmen.

------------------------------------------------
ORIGINAL:
Harvard University ist weltbekannt.

Resolving: Harvard University
QID: Q13371
Ro